# Predictive Analytics Using Historical Data
**Author:** Abhinav Gaur
**Dataset:** Superstore Sales (Kaggle: rohitsahoo/sales-forecasting)
**Goal:** Clean historical sales data, build predictive models to forecast future sales, evaluate their accuracy, and visualize the predictions.

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("train.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df = df.drop_duplicates()
print(df.shape)
print(df.isnull().sum()[["Order Date", "Sales"]])

(9800, 18)
Order Date    0
Sales         0
dtype: int64


## Data cleaning and preparation
The order dates were converted to a date format and duplicate rows were removed. Daily orders were then grouped into **monthly total sales**, giving a time series of 48 months (January 2015 to December 2018) with no missing months.

In [10]:
monthly = df.set_index("Order Date")["Sales"].resample("MS").sum().to_frame("Sales")
print(monthly.shape)       # should be 48 months
print(monthly.isnull().sum())
px.line(monthly, y="Sales", title="Monthly Sales (Historical Data)").show()

(48, 1)
Sales    0
dtype: int64


In [11]:
data = monthly.copy()
data["t"] = np.arange(len(data))        # trend
data["month"] = data.index.month        # seasonality

X = pd.get_dummies(data[["t", "month"]], columns=["month"], drop_first=True)
y = data["Sales"]

X_train, X_test = X.iloc[:-6], X.iloc[-6:]
y_train, y_test = y.iloc[:-6], y.iloc[-6:]
print(len(y_train), "training months,", len(y_test), "test months")

42 training months, 6 test months


## Historical patterns
- Yearly sales grew from about $480K in 2015 to about $722K in 2018, with a small dip in 2016.
- There is clear **seasonality**: sales peak in November and December, which together make up about 30% of annual sales. The highest month was November 2018 (about $118K).
- These two patterns, an upward trend and a yearly cycle, guide the choice of models.

In [12]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(X_train, y_train)
lr_pred = pd.Series(lr.predict(X_test), index=y_test.index)

## Feature engineering and train/test split
- A **trend** feature (month number) and a **month of year** feature (to capture seasonality) were created.
- The last 6 months were held out as the **test set**, and the model was trained on the earlier 42 months. This tests how well the model predicts months it has never seen.

## Models
1. **Linear Regression** using the trend and month-of-year features (regression approach).
2. **Holt-Winters Exponential Smoothing** with additive trend and seasonality (time-series approach).

In [13]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw = ExponentialSmoothing(y_train, trend="add", seasonal="add",
                          seasonal_periods=12).fit()
hw_pred = hw.forecast(6)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(name, actual, pred):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    return {"Model": name, "MAE": round(mae), "RMSE": round(rmse), "MAPE %": round(mape, 1)}

results = pd.DataFrame([
    evaluate("Linear Regression", y_test, lr_pred),
    evaluate("Holt-Winters", y_test, hw_pred),
])
results

,Model,MAE,RMSE,MAPE %
0,Linear Regression,14681,18498,17.7
1,Holt-Winters,13134,18420,15.8


## Model evaluation
The models were compared on the 6-month test set using three measures:
- **MAE:** average error in dollars
- **RMSE:** like MAE, but penalizes large errors more
- **MAPE:** average error as a percentage

| Model | MAE | RMSE | MAPE |
|---|---|---|---|
| Linear Regression | 14,681 | 18,498 | 17.7% |
| Holt-Winters | 13,134 | 18,420 | 15.8% |

**Holt-Winters performed slightly better** on all three measures, with a MAPE of 15.8% compared with 17.7% for Linear Regression, so it was used for the final forecast. The difference is small, especially in RMSE, which shows both models capture the overall trend and seasonality but miss some of the sharp month-to-month swings.

In [15]:
fig = go.Figure()
fig.add_scatter(x=y.index, y=y, name="Actual", mode="lines")
fig.add_scatter(x=lr_pred.index, y=lr_pred, name="Linear Regression", mode="lines+markers")
fig.add_scatter(x=hw_pred.index, y=hw_pred, name="Holt-Winters", mode="lines+markers")
fig.update_layout(title="Actual vs Predicted Monthly Sales (Test Period)",
                  xaxis_title="Month", yaxis_title="Sales")
fig.show()

## Actual vs predicted
Both models follow the overall pattern of the test period, including the large jump in September and the rise toward the end of the year. However, both **underestimate the November 2018 peak** (actual about $118K, predicted about $90K) and predict October too low. The two models give very similar predictions, which matches their similar error scores. These misses show that sudden spikes in sales are hard to predict from past patterns alone.

In [16]:
final = ExponentialSmoothing(y, trend="add", seasonal="add",
                             seasonal_periods=12).fit()
future = final.forecast(6)

fig = go.Figure()
fig.add_scatter(x=y.index, y=y, name="Historical", mode="lines")
fig.add_scatter(x=future.index, y=future, name="Forecast", mode="lines+markers",
                line=dict(dash="dash"))
fig.update_layout(title="6-Month Sales Forecast", xaxis_title="Month", yaxis_title="Sales")
fig.show()

future.round(0).to_frame("Forecast Sales")

,Forecast Sales
2019-01-01,49393.0
2019-02-01,41600.0
2019-03-01,71943.0
2019-04-01,60876.0
2019-05-01,68134.0
2019-06-01,62708.0


## Forecast
Using Holt-Winters retrained on all 48 months, sales were forecast for January to June 2019:

| Month | Forecast Sales |
|---|---|
| Jan 2019 | $49,393 |
| Feb 2019 | $41,600 |
| Mar 2019 | $71,943 |
| Apr 2019 | $60,876 |
| May 2019 | $68,134 |
| Jun 2019 | $62,708 |

The forecast continues the upward trend, with every month above the same month in 2018, and keeps the pattern of a weak start to the year followed by stronger months from March. Because the model underestimated sales peaks in the test period, the real figures could be higher in strong months.

## Conclusion and recommendations
- Sales have an upward trend with strong end-of-year seasonality, so forecasts should account for both.
- [Model name] gave the most accurate predictions on unseen data, with an average error of about [x]%.
- The business can use the forecast to plan **inventory, staffing and marketing**, especially ahead of November and December.

**Limitations:** the dataset covers only 4 years, the test set has just 6 months, and the models do not include outside factors such as promotions, pricing, or economic conditions. More data and extra features would improve the accuracy.

**Tools used:** Python, Pandas, scikit-learn, statsmodels, Plotly.